In [38]:
from pyspark.sql import SparkSession
spark = (SparkSession. 
builder.
master("spark://spark-master:7077")
.appName("dataframe").getOrCreate()
        )

In [37]:
spark.stop()

In [31]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
orders_schema_struct = StructType(
[
StructField("order_id",IntegerType(),False)
,StructField("customer_id",IntegerType(),False)
,StructField("product_id",IntegerType(),False)
,StructField("unit_price",FloatType(),False)
,StructField("order_date",DateType(),False)
,StructField("order_status",StringType(),False)
,StructField("state",StringType(),False)
,StructField("quantity",IntegerType(),False)
]   
)

df_orders = spark.read.csv("/data/orders_1gb.csv" , header = True ,  schema = orders_schema_struct)

product_schema =  'product_id int , cost_price float'

df_products = spark.read.csv("/data/products_12mb.csv" , header = True ,  schema = product_schema)
df_orders.show()
df_products.show()

+--------+-----------+----------+----------+----------+------------+--------------+--------+
|order_id|customer_id|product_id|unit_price|order_date|order_status|         state|quantity|
+--------+-----------+----------+----------+----------+------------+--------------+--------+
|       1|     997568|        16|    148.27|2022-09-19|     SHIPPED|         Assam|       1|
|       2|     980660|        90|    193.15|2019-07-20|     SHIPPED|Madhya Pradesh|       1|
|       3|     293603|        47|    117.13|2006-11-16|      PLACED|         Delhi|       2|
|       4|      88784|        40|    169.87|2021-12-20|   DELIVERED|     Karnataka|       3|
|       5|     639307|        89|    125.48|2022-12-12|   DELIVERED|     Jharkhand|       1|
|       6|     460431|        83|     179.2|2000-01-17|     SHIPPED|     Rajasthan|       1|
|       7|     197087|        14|    185.22|2019-09-21|     SHIPPED|        Punjab|       3|
|       8|     881816|         8|    161.35|2012-10-13|      PLACED|  

In [12]:
basics of spark architecture , SAPRk UI
daframe API, spark sql
performace 

In [25]:
df_products_filtered = df_products.filter(col("product_id")<=1000)
df_products_filtered.show()

In [32]:
df_orders_filtered = df_orders.filter(col("product_id")<=1000)
df_orders_filtered.write.format("noop").mode("overwrite").save()

In [12]:
df_joined = df_orders.join(df_products, "product_id" , "inner")
df_joined.write.format("noop").mode("overwrite").save()

In [31]:
spark.conf.set("spark.sql.adaptive.enabled",True)

In [ ]:
orders 
products -> filter -> 100 (<10mb) broadcast ->

In [29]:
df_skewed = df_orders.withColumn("product_id",  when(col("product_id")<=90 , 1 ).otherwise(col("product_id")))
df_skewed.groupBy("product_id").count().show()

+----------+--------+
|product_id|   count|
+----------+--------+
|        91|  203774|
|        93|  204122|
|         1|18367073|
|        94|  204212|
|        96|  203548|
|        92|  203557|
|        97|  204495|
|        95|  204639|
|        98|  204580|
+----------+--------+



In [32]:
df_joined = df_skewed.join(df_products, "product_id" , "inner")
df_joined.write.format("noop").mode("overwrite").save()

In [ ]:
aggregation 
group_by product_id 
18 million -> 1 partion -> p1, 18000000

p1,p1_key1 , 1.8 m
p1,p1_key2 , 1.8 m
p1,p1_key3 , 1.8 m
p1,p1_key4 , 1.8 m

customer 

product_id 18 million records
P1
1.8 million records -> P1 -> custom logic ()
1.8 million records -> P1
1.8 million records -> P1
1.8 million records -> P1

1.8 million records -> P1_key10

Salting

spark 3.0
orders (20 million)
order_id, order_date, product
1 , 2026-01-01 , p1
2 , 2026-01-01 , p1

P1 -> 18 million 60 min -> 6mins
2 million 5 min
p1 -> 

1 big job -> 

file 1-> job1 -> filter -> with column (data)
file 2 -> job2  -> narrow  (data)

shuffle 

In [ ]:
orders
p1
p2
p3  shuffle ->  18 milion -> join products 



In [35]:
!spark-submit \
--master spark://spark-master:7077 \
--deploy-mode client \
job.py

26/01/31 15:00:57 INFO SparkContext: Running Spark version 3.5.1
26/01/31 15:00:57 INFO SparkContext: OS info Linux, 6.12.54-linuxkit, aarch64
26/01/31 15:00:57 INFO SparkContext: Java version 17.0.15
26/01/31 15:00:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/31 15:00:57 INFO ResourceUtils: ==============================================================
26/01/31 15:00:57 INFO ResourceUtils: No custom resources configured for spark.driver.
26/01/31 15:00:57 INFO ResourceUtils: ==============================================================
26/01/31 15:00:57 INFO SparkContext: Submitted application: dataframe
26/01/31 15:00:57 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus, amount: 1.0)
26/01/31 15:00:57 INFO Resource

In [ ]:
spark-submit \
  --executor-memory 4g \
  --executor-cores 2 \
  --conf spark.sql.shuffle.partitions=10 \
  app.py

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession. 
builder.
master("spark://spark-master:7077")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.dynamicAllocation.minExecutors", "1")
    .config("spark.dynamicAllocation.initialExecutors", "1")
    .config("spark.dynamicAllocation.maxExecutors", "4")
    .config("spark.dynamicAllocation.schedulerBacklogTimeout","5s")
    .config("spark.dynamicAllocation.sustainedSchedulerBacklogTimeout", "2s")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
.appName("dataframe").getOrCreate()
        )

In [ ]:
# notebook
!spark-submit \
--master spark://spark-master:7077 \
--deploy-mode client \
job.py


# terminal 
docker exec -it spark-master bash
# create job.py in master container 
/opt/spark/bin/spark-submit --master spark://spark-master:7077 --deploy-mode client job.py

In [ ]:
order_id, product_id , price , customer_id, customer_name, customer_email
1 , P1 , 100
1,  P2, 200

{"order_id":1 , "product_id" : "P1" , price : 100 , customer :{id:1,name:"ankit",email:"abc@gmail.com" } }
{"order_id":1 , products : [[P1] ,P2] , price : 100 , customer :{id:1,name:"ankit",email:"abc@gmail.com" } }


{"order_id":2 , "product_id" : "P2" , price : 200}

document -> no sql 
api response -> data 

In [39]:
df_json = spark.read.json("/data/orders.json")

In [41]:
df_json.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- sales: double (nullable = true)
 |-- state: string (nullable = true)
 |-- status: string (nullable = true)



In [57]:
df_json = spark.read.json("/data/orders_nested.json", schema = orders_schema)

In [58]:
df_json.show()

+--------+------------+--------------------+
|order_id|    customer|               items|
+--------+------------+--------------------+
|       1|{101, Ankit}|[{Laptop, 50000},...|
|       2|{102, Rahul}|   [{Mobile, 20000}]|
+--------+------------+--------------------+



In [45]:
df = df_json.select("customer.customer_id", "customer.name", "order_date", "order_id", "items")

In [55]:
df_json.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer: struct (nullable = true)
 |    |-- id: integer (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product: string (nullable = true)
 |    |    |-- price: integer (nullable = true)



In [48]:
df_json.printSchema()

root
 |-- customer: struct (nullable = true)
 |    |-- customer_id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: long (nullable = true)
 |    |    |-- product: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)



In [51]:
df_flatten = df.withColumn("flat_items" , explode("items") ).drop("items")
df_flatten.select("customer_id","name","order_date","order_id", "flat_items.price","flat_items.product").show()

+-----------+-----+----------+--------+-----+-------+
|customer_id| name|order_date|order_id|price|product|
+-----------+-----+----------+--------+-----+-------+
|        101|Ankit|2025-01-11|       1|50000| Laptop|
|        101|Ankit|2025-01-11|       1|  500|  Mouse|
|        102|Rahul|2025-01-12|       2|20000| Mobile|
+-----------+-----+----------+--------+-----+-------+



In [56]:
orders_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer", StructType([
        StructField("customer_id", IntegerType()),
        StructField("name", StringType())
    ])),
    StructField("items", ArrayType(
        StructType([
            StructField("product", StringType()),
            StructField("price", IntegerType())
        ])
    ))
])

In [ ]:
df.write.mode("overwrite") order=1 , order_date
p0.parquet
p1.parquet